# 🤖 Modeling & Visualization

**Business Story**: Meet Customer Sarah, a 45-year-old customer from Germany with a balance of $120,000 and only one product. Our model predicts she has a 73% chance of churning within the next quarter.

**Action**: Priority call from relationship manager + personalized product offer.

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
sys.path.append('..')

from sklearn.metrics import roc_auc_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid')

## Load Data & Models

In [ ]:
# load data
X_train, X_val, X_test, y_train, y_val, y_test = pd.read_pickle('../data/processed/features.pkl')
print(f'Test set: {X_test.shape}')

In [ ]:
# load models
models = {}
model_names = ['logreg', 'rf', 'lgbm', 'xgb', 'catboost', 'stacking']
for name in model_names:
    try:
        models[name] = joblib.load(f'../models/{name}.pkl')
    except:
        pass
print(f'Loaded models: {list(models.keys())}')

## Model Comparison

Comparing model performance to select best approach.

In [ ]:
# compare models
results = {}
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    results[name] = auc

results_df = pd.DataFrame({'Model': results.keys(), 'ROC-AUC': results.values()})
results_df = results_df.sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
# plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(results_df)))
ax.barh(results_df['Model'], results_df['ROC-AUC'], color=colors)
ax.set_xlabel('ROC-AUC')
ax.set_title('Model Performance Comparison')
ax.set_xlim(0.5, 1.0)
for i, v in enumerate(results_df['ROC-AUC']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig('../figs/model_comparison.png', dpi=150)

## ROC & PR Curves

In [ ]:
# use best model
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]
y_proba = best_model.predict_proba(X_test)[:, 1]
print(f'Best model: {best_model_name}')

In [ ]:
# roc curve
from src.evaluate import plot_roc, plot_pr_curve, plot_calibration, plot_lift_curve, plot_gain_curve

plot_roc(y_test, y_proba, '../figs/roc.png')
plot_pr_curve(y_test, y_proba, '../figs/pr_curve.png')
plot_calibration(y_test, y_proba, out='../figs/calibration.png')
plot_lift_curve(y_test, y_proba, '../figs/lift.png')
plot_gain_curve(y_test, y_proba, '../figs/gain.png')
print('Plots saved.')

## Feature Importance

Understanding what drives churn predictions.

In [ ]:
# feature importance
from src.explain import plot_feature_importance
plot_feature_importance(best_model, list(X_test.columns), '../figs/feature_importance.png')
print('Feature importance saved.')

## Business Metrics

Translating model performance to business value.

In [ ]:
# business metrics
from src.evaluate import precision_at_k, lift_at_k

metrics = {
    'Precision@1%': precision_at_k(y_test, y_proba, 0.01),
    'Precision@5%': precision_at_k(y_test, y_proba, 0.05),
    'Precision@10%': precision_at_k(y_test, y_proba, 0.10),
    'Lift@5%': lift_at_k(y_test, y_proba, 0.05),
    'Lift@10%': lift_at_k(y_test, y_proba, 0.10)
}

for k, v in metrics.items():
    print(f'{k}: {v:.2f}')

## ROI Simulation

**Assumptions**:
- Customer lifetime value: $2,000
- Retention campaign cost: $50 per customer
- Retention success rate: 30%

In [ ]:
# roi simulation
CLTV = 2000
COST_PER_CUSTOMER = 50
RETENTION_RATE = 0.30
TOTAL_CUSTOMERS = 10000

results_roi = []
for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
    n_targeted = int(TOTAL_CUSTOMERS * pct)
    prec = precision_at_k(y_test, y_proba, pct)
    true_churners = int(n_targeted * prec)
    saved = int(true_churners * RETENTION_RATE)
    revenue_saved = saved * CLTV
    cost = n_targeted * COST_PER_CUSTOMER
    roi = (revenue_saved - cost) / cost if cost > 0 else 0
    results_roi.append({
        'Target%': f'{pct:.0%}',
        'Targeted': n_targeted,
        'Churners': true_churners,
        'Saved': saved,
        'Revenue': f'${revenue_saved:,}',
        'Cost': f'${cost:,}',
        'ROI': f'{roi:.1%}'
    })

pd.DataFrame(results_roi)

In [ ]:
# plot roi
pcts = [0.01, 0.05, 0.10, 0.20, 0.30]
rois = []
for pct in pcts:
    n_targeted = int(TOTAL_CUSTOMERS * pct)
    prec = precision_at_k(y_test, y_proba, pct)
    true_churners = int(n_targeted * prec)
    saved = int(true_churners * RETENTION_RATE)
    revenue_saved = saved * CLTV
    cost = n_targeted * COST_PER_CUSTOMER
    roi = (revenue_saved - cost) / cost if cost > 0 else 0
    rois.append(roi)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([f'{p:.0%}' for p in pcts], rois, color='#2ecc71')
ax.set_xlabel('% Population Targeted')
ax.set_ylabel('ROI')
ax.set_title('Expected ROI by Campaign Size')
ax.axhline(y=0, color='gray', linestyle='--')
plt.tight_layout()
plt.savefig('../figs/roi_simulation.png', dpi=150)

## Top Risk Customers

Priority list for retention team.

In [ ]:
# top risk customers
X_test_copy = X_test.copy()
X_test_copy['churn_probability'] = y_proba
X_test_copy['actual_churn'] = y_test.values
top_risk = X_test_copy.nlargest(10, 'churn_probability')[['churn_probability', 'actual_churn']]
top_risk

## Recommendations

1. **Target top 5%** of predicted churners for maximum ROI
2. **Focus on Germany** — highest churn geography
3. **Cross-sell** to single-product customers
4. **Engage** customers with zero balance proactively
5. **Monitor** model drift quarterly